In [1]:
%pip install requests beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


#### 문항 1. 할리스커피 매장 정보 10페이지 수집

할리스커피 매장 찾기 페이지에서 매장 정보를 수집해 리스트-딕셔너리 형태로 정리하시오.

* 대상: https://www.hollys.co.kr/store/korea/korStore2.do

* 총 10페이지를 순회할 것 (페이지를 넘기며 URL이 바뀌는 것을 확인)

* 추출 필드: 지역 / 매장명 / 현황 / 주소 / 매장 서비스 / 전화번호

* 매장 서비스는 리스트로 담을 것 (아이콘이 여러 개인 매장이 있음)

* 결과를 hollys.csv로 저장할 것
```
# 결과 예시
[{'지역': '서울 동대문구',
  '매장명': '경희대 경영대점',
  '현황': '영업중',
  '주소': '서울특별시 동대문구 경희대로 26 (회기동) 경영대학 3층',
  '매장 서비스': ['주차'],
  '전화번호': '.'},
 ...]
```

In [ ]:
import csv
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.hollys.co.kr/store/korea/korStore2.do"
HEADERS = {"User-Agent": "Mozilla/5.0 "}

def get_store_list(total_pages: int = 10):
    result = []

    for page in range(1, total_pages + 1):
        params = {
            "pageNo": page,
            "sido": "",
            "gugun": "",
            "store": "",
        }
    
        res = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=10)
        res.encoding = "utf-8"

        soup = BeautifulSoup(res.text, "html.parser")

        tbody = soup.find("tbody")
        if tbody is None:
            print(f"{page} 페이지에서 표(tbody)를 찾지 못했어요.")
            continue

        rows = tbody.find_all("tr")
        page_count = 0

        for row in rows:
            cols = row.find_all("td")
            if len(cols) < 6:
                continue

            area = cols[0].get_text(strip=True)      
            name = cols[1].get_text(strip=True)      
            status = cols[2].get_text(strip=True)      
            address = cols[3].get_text(strip=True)      

            services = [
                img.get("alt", "").strip()
                for img in cols[4].find_all("img")
                if img.get("alt", "").strip()
            ]

            phone = cols[5].get_text(strip=True)       

            result.append(
                {
                    "지역": area,
                    "매장명": name,
                    "현황": status,
                    "주소": address,
                    "매장 서비스": services,
                    "전화번호": phone,
                }
            )
            page_count += 1

        print(f"{page} 페이지 완료 - {page_count}개 매장 수집 (누적 {len(result)}개)")
    return result

def save_to_csv(data, filename: str = "hollys.csv"):
    fieldnames = ["지역", "매장명", "현황", "주소", "매장 서비스", "전화번호"]

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in data:
            row_to_write = row.copy()
            row_to_write["매장 서비스"] = ", ".join(row["매장 서비스"])
            writer.writerow(row_to_write)

if __name__ == "__main__":
    stores = get_store_list(total_pages=10)
    save_to_csv(stores, "hollys.csv")
    print(f"\n총 {len(stores)}개 매장 정보를 hollys.csv 파일로 저장했습니다.")

1 페이지 완료 - 10개 매장 수집 (누적 10개)
2 페이지 완료 - 10개 매장 수집 (누적 20개)
3 페이지 완료 - 10개 매장 수집 (누적 30개)
4 페이지 완료 - 10개 매장 수집 (누적 40개)
5 페이지 완료 - 10개 매장 수집 (누적 50개)
6 페이지 완료 - 10개 매장 수집 (누적 60개)
7 페이지 완료 - 10개 매장 수집 (누적 70개)
8 페이지 완료 - 10개 매장 수집 (누적 80개)
9 페이지 완료 - 10개 매장 수집 (누적 90개)
10 페이지 완료 - 10개 매장 수집 (누적 100개)

총 100개 매장 정보를 hollys.csv 파일로 저장했습니다.


#### 문항 2. 알라딘 베스트셀러 수집

알라딘 베스트셀러 페이지에서 도서 정보를 수집해 CSV로 저장하시오.

* 대상: 알라딘 베스트셀러 목록 페이지(https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1)

* 추출 필드: 카테고리 / 제목 / 저자 / 할인가격 / 이미지 URL

* 이미지 URL은 <img> 태그의 속성에서 가져올 것

* 결과를 aladin_bestseller.csv로 저장할 것

* 추가학습: 총 500위까지 수집해주세요.

결과 예시 (8월 2주)
```
[{'카테고리': '[국내도서]',
  '제목': '오뒷세이아',
  '저자': '호메로스',
  '정가': '25,000원',
  '할인가격': '22,500원',
  '이미지': '<https://image.aladin.co.kr/product/39940/12/cover200/8932476462_2.jpg>'},
 ...]
```

In [ ]:
import csv
import re
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.aladin.co.kr/shop/common/wbest.aspx"
HEADERS = {"User-Agent": "Mozilla/5.0 "}
PRICE_PATTERN = re.compile(r"([\d,]+)\s*원\s*(?:→|->)\s*([\d,]+)\s*원")
CATEGORY_PATTERN = re.compile(r"\[([^\[\]]+)\]")

def find_outer_row(book_box):
    node = book_box
    for _ in range(10):
        node = node.find_parent("tr")
        if node is None:
            return None
        tds = node.find_all("td", recursive=False)
        if len(tds) > 1:
            return node
    return node

def extract_image_url(row):
    if row is None:
        return ""

    preview_link = row.find("a", href=re.compile(r"fn_PopUpPriview"))
    img = preview_link.find("img") if preview_link else row.find("img")
    if img is None:
        return ""

    candidate_attrs = ("src", "data-original", "data-src", "ec-data-src", "data-lazy")
    for attr in candidate_attrs:
        value = img.get(attr)
        if value and "blank.gif" not in value:
            if value.startswith("//"):
                value = "https:" + value
            elif value.startswith("/"):
                value = "https://www.aladin.co.kr" + value
            return value
    return ""

def parse_page(html_text: str):
    soup = BeautifulSoup(html_text, "html.parser")
    result = []

    book_boxes = soup.find_all("div", class_="ss_book_list")

    for box in book_boxes:
        title_tag = box.find("a", class_="bo3")
        if title_tag is None:
            continue
        title = title_tag.get_text(strip=True)

        full_text = box.get_text(" ", strip=True)

        cat_match = CATEGORY_PATTERN.search(full_text)
        category = f"[{cat_match.group(1)}]" if cat_match else ""

        price_match = PRICE_PATTERN.search(full_text)
        if price_match:
            regular_price = price_match.group(1) + "원"
            sale_price = price_match.group(2) + "원"
        else:
            regular_price = ""
            sale_price = ""

        after_title = full_text.split(title, 1)[-1]
        author = after_title.split("|", 1)[0].strip()

        outer_row = find_outer_row(box)
        image_url = extract_image_url(outer_row)

        result.append(
            {
                "카테고리": category,
                "제목": title,
                "저자": author,
                "정가": regular_price,
                "할인가격": sale_price,
                "이미지": image_url,
            }
        )
    return result

def get_bestseller(total_rank: int = 500, per_page: int = 50):
    total_pages = total_rank // per_page  # 500 // 50 = 10페이지
    result = []

    for page in range(1, total_pages + 1):
        params = {
            "BestType": "Bestseller",
            "BranchType": 1,
            "CID": 0,
            "page": page,
            "cnt": 1000,
            "SortOrder": 1,
        }
        res = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=10)
        res.encoding = "utf-8"

        page_items = parse_page(res.text)
        result.extend(page_items)

        print(f"{page} 페이지 완료 - {len(page_items)}권 수집 (누적 {len(result)}권)")
        time.sleep(0.3)

    return result[:total_rank]

def save_to_csv(data, filename: str = "aladin_bestseller.csv"):
    fieldnames = ["카테고리", "제목", "저자", "정가", "할인가격", "이미지"]
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in data:
            writer.writerow(row)

if __name__ == "__main__":
    books = get_bestseller(total_rank=500)
    save_to_csv(books)
    print(f"\n총 {len(books)}권의 도서 정보를 aladin_bestseller.csv 파일로 저장했습니다.")

1 페이지 완료 - 50권 수집 (누적 50권)
2 페이지 완료 - 50권 수집 (누적 100권)
3 페이지 완료 - 50권 수집 (누적 150권)
4 페이지 완료 - 50권 수집 (누적 200권)
5 페이지 완료 - 50권 수집 (누적 250권)
6 페이지 완료 - 50권 수집 (누적 300권)
7 페이지 완료 - 50권 수집 (누적 350권)
8 페이지 완료 - 50권 수집 (누적 400권)
9 페이지 완료 - 50권 수집 (누적 450권)
10 페이지 완료 - 50권 수집 (누적 500권)

총 500권의 도서 정보를 aladin_bestseller.csv 파일로 저장했습니다.
